<a href="https://colab.research.google.com/github/Hariom1-eng/Deep-Learning/blob/main/Recurrent_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Recurrent Neural Networks (RNNs), LSTMs, and GRUs

**Recurrent Neural Networks (RNNs)** are a class of neural networks designed to recognize patterns in sequences of data, such as text, speech, or time series. Unlike traditional feedforward networks, RNNs have internal memory, allowing them to use information from previous inputs in the sequence to influence the current output.

**Long Short-Term Memory (LSTM)** networks are a special kind of RNN, capable of learning long-term dependencies. They were introduced to combat the vanishing gradient problem that plagues traditional RNNs, enabling them to remember information for extended periods. LSTMs achieve this through a complex gating mechanism (input gate, forget gate, output gate) that regulates the flow of information.

**Gated Recurrent Units (GRUs)** are a slightly simplified version of LSTMs. They combine the forget and input gates into a single "update gate" and also merge the cell state and hidden state. GRUs are generally faster to train and computationally less expensive than LSTMs, while often achieving similar performance on various tasks.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Prepare Sample Data
# Let's create a simple sequence prediction task
data = [
    "The quick brown fox jumps over the lazy dog",
    "A quick brown fox",
    "Jumps over the dog",
    "The lazy dog barks",
    "Fox is quick"
]

# Create sequences of words (simple next-word prediction)
sequences = []
for line in data:
    words = line.lower().split()
    for i in range(1, len(words)):
        sequence = words[:i+1]
        sequences.append(sequence)

print("Sample sequences:")
for seq in sequences[:5]:
    print(seq)

# 2. Tokenize and Pad Sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print(f"\nVocabulary size: {vocab_size}")
print("Word index:", word_index)

input_sequences = []
for seq in sequences:
    encoded_seq = [word_index[word] for word in seq]
    input_sequences.append(encoded_seq)

max_sequence_len = max([len(x) for x in input_sequences])
print(f"\nMaximum sequence length: {max_sequence_len}")

padded_sequences = pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')

X = padded_sequences[:, :-1] # All but the last word
y = padded_sequences[:, -1] # The last word (target)

# Convert y to one-hot encoding for categorical crossentropy
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

print("\nShape of X (input sequences):", X.shape)
print("Shape of y (target words - one-hot):", y.shape)

# 3. Build the LSTM Model
model = Sequential()
model.add(Embedding(vocab_size, 10)) # Embedding layer
model.add(LSTM(50)) # LSTM layer with 50 units
model.add(Dense(vocab_size, activation='softmax')) # Output layer with softmax for word prediction

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\nModel Summary:")
model.summary()

# 4. Train the Model (for demonstration, a small number of epochs)
print("\nTraining the model...")
model.fit(X, y, epochs=50, verbose=0) # Set verbose to 1 for progress

# 5. Make a Prediction
def predict_next_word(model, tokenizer, max_len, text):
    token_list = tokenizer.texts_to_sequences([text])[0]
    padded_token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predicted_probs = model.predict(padded_token_list, verbose=0)
    predicted_word_index = np.argmax(predicted_probs)

    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None


seed_text = "the quick brown"
next_word = predict_next_word(model, tokenizer, max_sequence_len, seed_text)
print(f"\nGiven '{seed_text}', the predicted next word is: {next_word}")

seed_text = "jumps over"
next_word = predict_next_word(model, tokenizer, max_sequence_len, seed_text)
print(f"Given '{seed_text}', the predicted next word is: {next_word}")

Sample sequences:
['the', 'quick']
['the', 'quick', 'brown']
['the', 'quick', 'brown', 'fox']
['the', 'quick', 'brown', 'fox', 'jumps']
['the', 'quick', 'brown', 'fox', 'jumps', 'over']

Vocabulary size: 12
Word index: {'the': 1, 'quick': 2, 'fox': 3, 'dog': 4, 'brown': 5, 'jumps': 6, 'over': 7, 'lazy': 8, 'a': 9, 'barks': 10, 'is': 11}

Maximum sequence length: 9

Shape of X (input sequences): (19, 8)
Shape of y (target words - one-hot): (19, 12)

Model Summary:


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the model...

Given 'the quick brown', the predicted next word is: quick
Given 'jumps over', the predicted next word is: quick


### Gated Recurrent Unit (GRU) Model Example

**Gated Recurrent Units (GRUs)** are a simpler variant of LSTMs. They combine the forget and input gates into a single "update gate" and also merge the cell state and hidden state. This makes them computationally less expensive and faster to train than LSTMs, while often offering comparable performance.

In [ ]:
from tensorflow.keras.layers import GRU

# 1. Build the GRU Model
# We'll use the same tokenizer, vocab_size, max_sequence_len, X, and y from the previous example.

gru_model = Sequential()
gru_model.add(Embedding(vocab_size, 10)) # Embedding layer
gru_model.add(GRU(50)) # GRU layer with 50 units
gru_model.add(Dense(vocab_size, activation='softmax')) # Output layer with softmax for word prediction

gru_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\nGRU Model Summary:")
gru_model.summary()

# 2. Train the GRU Model
print("\nTraining the GRU model...")
gru_model.fit(X, y, epochs=50, verbose=0) # Using the same X and y

# 3. Make a Prediction with GRU model
seed_text_gru = "the quick brown"
next_word_gru = predict_next_word(gru_model, tokenizer, max_sequence_len, seed_text_gru)
print(f"\nGiven '{seed_text_gru}', the GRU predicted next word is: {next_word_gru}")

seed_text_gru = "jumps over"
next_word_gru = predict_next_word(gru_model, tokenizer, max_sequence_len, seed_text_gru)
print(f"Given '{seed_text_gru}', the GRU predicted next word is: {next_word_gru}")


GRU Model Summary:


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the GRU model...



Given 'the quick brown', the GRU predicted next word is: quick
Given 'jumps over', the GRU predicted next word is: dog


### Simple Recurrent Neural Network (RNN) Model Example

**SimpleRNN** is the vanilla form of a Recurrent Neural Network. It processes sequences one element at a time, maintaining an internal state (hidden state) that is updated with each new input. While foundational, SimpleRNNs often suffer from the vanishing/exploding gradient problem, making it difficult for them to learn long-term dependencies compared to LSTMs and GRUs.

In [ ]:
from tensorflow.keras.layers import SimpleRNN

# 1. Build the SimpleRNN Model
# We'll use the same tokenizer, vocab_size, max_sequence_len, X, and y.

rnn_model = Sequential()
rnn_model.add(Embedding(vocab_size, 10)) # Embedding layer
rnn_model.add(SimpleRNN(50)) # SimpleRNN layer with 50 units
rnn_model.add(Dense(vocab_size, activation='softmax')) # Output layer for word prediction

rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\nSimpleRNN Model Summary:")
rnn_model.summary()

# 2. Train the SimpleRNN Model
print("\nTraining the SimpleRNN model...")
rnn_model.fit(X, y, epochs=50, verbose=0) # Using the same X and y

# 3. Make a Prediction with SimpleRNN model
seed_text_rnn = "the quick brown"
next_word_rnn = predict_next_word(rnn_model, tokenizer, max_sequence_len, seed_text_rnn)
print(f"\nGiven '{seed_text_rnn}', the SimpleRNN predicted next word is: {next_word_rnn}")

seed_text_rnn = "jumps over"
next_word_rnn = predict_next_word(rnn_model, tokenizer, max_sequence_len, seed_text_rnn)
print(f"Given '{seed_text_rnn}', the SimpleRNN predicted next word is: {next_word_rnn}")


SimpleRNN Model Summary:


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


Training the SimpleRNN model...



Given 'the quick brown', the SimpleRNN predicted next word is: fox
Given 'jumps over', the SimpleRNN predicted next word is: quick


### Bidirectional Encoder Representations from Transformers (BERT) Model Example

**BERT (Bidirectional Encoder Representations from Transformers)** is a powerful, pre-trained language model developed by Google. Unlike traditional recurrent neural networks that process text sequentially, BERT processes the entire sequence at once, making it "bidirectional." This allows it to understand the context of a word based on all other words in a sentence, both to its left and right.

BERT is pre-trained on a massive amount of text data using two unsupervised tasks: Masked Language Model (MLM) and Next Sentence Prediction (NSP). This pre-training enables it to learn rich, contextual representations of words, which can then be fine-tuned for a wide range of downstream natural language processing (NLP) tasks like sentiment analysis, question answering, and text summarization. The `transformers` library by Hugging Face provides an easy way to access and use BERT and many other state-of-the-art models.